# The Memory-Efficient Data Pipeline - Iterators and Generators

Difficulty: Intermediate | ~40 min | Requires Lab 1 basics

One 50 GB log file, one laptop with 16 GB of RAM, and a deadline. Reading the
file into a list is not an option - this lab builds a memory-safe reader two
ways (a class-based iterator and a generator function) and finds error codes
and traffic spikes without ever loading the whole file.

In [1]:
!pip install tabulate==0.10.0

## Section 0 - Warm up: the iterator protocol

Run the cell to feel `iter()`, `next()`, and `StopIteration`.

In [2]:
# Warm up: iter() hands you an iterator, next() pulls one item, and
# StopIteration marks the end of the stream.
numbers = iter([10, 20, 30])
print(next(numbers))
print(next(numbers))
print(next(numbers))
try:
    print(next(numbers))
except StopIteration:
    print("StopIteration - the iterator is exhausted")

10
20
30
StopIteration - the iterator is exhausted


## Section 1 - The scenario

A traffic spike hits a 50 GB log; count `500`s and spikes without loading the file into RAM.

In [3]:
# Generate a deterministic fake server log - no downloads, no API keys.
# random.seed(42) makes every learner produce the identical file.
import os
import random

random.seed(42)
os.makedirs("data", exist_ok=True)
paths = ["/api/items", "/api/users", "/api/orders", "/static/app.js"]
with open("data/server.log", "w", encoding="utf-8") as f:
    # One hour of traffic; each second writes `count` request lines.
    for second in range(3600):
        burst = random.random() < 0.02
        count = random.randint(150, 500) if burst else random.randint(5, 60)
        for _ in range(count):
            # One log line: date time method path status.
            hh, mm, ss = second // 3600, second % 3600 // 60, second % 60
            stamp = f"2026-08-09 {hh:02d}:{mm:02d}:{ss:02d}"
            path = random.choice(paths)
            method = random.choice(["GET", "GET", "POST"])
            status = random.choice(["200", "200", "200", "404", "500"])
            f.write(f"{stamp} {method} {path} {status}\n")
print("data/server.log written:", os.path.getsize("data/server.log"), "bytes")

data/server.log written: 5846826 bytes


## Section 2 - Our data

Generate a deterministic `data/server.log` (`date time method path status` per line) so everyone gets the identical file.

In [4]:
# Baseline: read the whole file into one list of strings.
import sys

with open("data/server.log", encoding="utf-8") as f:
    lines_list = f.readlines()
# Every line is now a string in memory at once - the thing we'll shrink.
print("Lines read at once:", len(lines_list))
print("sys.getsizeof(list):", sys.getsizeof(lines_list), "bytes")

Lines read at once: 140610
sys.getsizeof(list): 1140568 bytes


## Section 3 - The naive approach

Baseline: `readlines()` loads the whole file into one list - correct, but that size number is the thing we will shrink.

In [5]:
# Part 1 - a class-based iterator. Only chunk_size lines live in memory.
class LogReader:
    def __init__(self, path, chunk_size=1000):
        self.path = path
        self.chunk_size = chunk_size
        self.file = None
        self.buffer = []
        self.index = 0

    def __iter__(self):
        # The for loop calls this first: open the file and reset state.
        self.file = open(self.path, encoding="utf-8")
        self.buffer = []
        self.index = 0
        return self

    def __next__(self):
        # Called for every loop iteration; returns the next line.
        if self.index == len(self.buffer):
            # Buffer is empty: refill it with up to chunk_size lines.
            self.buffer = []
            for _ in range(self.chunk_size):
                line = self.file.readline()
                if not line:  # EOF - no more lines to read.
                    break
                self.buffer.append(line)
            self.index = 0
            if not self.buffer:
                # File exhausted: close it and signal the loop to stop.
                self.file.close()
                raise StopIteration
        line = self.buffer[self.index]
        self.index += 1
        return line

## Section 4 - Part 1: a class-based iterator

Implement `__iter__` (return self) and `__next__` (next value or `StopIteration`), reading in chunks so only a slice is in memory.

In [6]:
# Same for loop, but only one chunk of lines is in memory at a time.
total_class = 0
errors_class = 0
for line in LogReader("data/server.log"):
    total_class += 1
    if line.split()[-1] == "500":  # status code is the last field
        errors_class += 1
print("Total lines (class iterator):", total_class)
print("500 errors (class iterator):", errors_class)

Total lines (class iterator): 140610
500 errors (class iterator): 28321


## Section 5 - Part 1: use it

Drop `LogReader` into a normal `for` loop and count total lines and `500` errors.

In [7]:
# Part 2 - the same reader as a generator function. Calling it returns a
# generator object; no file is read until you iterate.
def read_log_lines(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield line  # hands back one line and pauses here

## Section 6 - Part 2: the generator

Same reader as a generator function - `yield` hands back a value and suspends, so the boilerplate disappears.

In [8]:
# Verify the generator agrees with the class iterator. Same counting loop
# as Part 1 - only the reader object changed.
total_gen = 0
errors_gen = 0
for line in read_log_lines("data/server.log"):
    total_gen += 1
    if line.split()[-1] == "500":
        errors_gen += 1
print("Total lines (generator):", total_gen)
print("500 errors (generator):", errors_gen)
print("Totals match:", (total_gen, errors_gen) == (total_class, errors_class))

Total lines (generator): 140610
500 errors (generator): 28321
Totals match: True


## Section 7 - Part 2: verify, then find spikes

Confirm the generator matches the class, then use `groupby` to find the busiest and spike seconds.

In [9]:
# Traffic spikes: groupby groups consecutive lines by their full timestamp
# (date + time); sum counts each group without storing it. Generators are
# single-pass, so re-call to re-read.
from itertools import groupby

def seconds_and_counts(lines):
    # " ".join(line.split()[:2]) is the timestamp (date + time), so seconds
    # never collide across days even if the log grew.
    for ts, group in groupby(lines, key=lambda line: " ".join(line.split()[:2])):
        yield ts, sum(1 for _ in group)

print("Busiest second:", max(seconds_and_counts(read_log_lines("data/server.log")),
                             key=lambda kv: kv[1]))
n_spikes = sum(1 for ts, n in seconds_and_counts(read_log_lines("data/server.log")) if n > 100)
print("Seconds with > 100 requests:", n_spikes)

Busiest second: ('2026-08-09 00:13:38', 500)


Seconds with > 100 requests: 76


## Section 8 - The memory payoff

`sys.getsizeof()` the list vs the generator - same answers, one is thousands of times lighter.

In [10]:
# The payoff, measured shallow: the list owns a pointer to every line, while
# the generator object owns just a file handle and a saved position. The
# exact byte counts vary by Python version/environment.
import sys

gen = read_log_lines("data/server.log")
list_size = sys.getsizeof(lines_list)
gen_size = sys.getsizeof(gen)
print("readlines() list object:", list_size, "bytes")
print("generator object:", gen_size, "bytes")
print("Ratio:", round(list_size / gen_size, 1), "x")

readlines() list object: 1140568 bytes
generator object: 232 bytes
Ratio: 4916.2 x


## Section 9 - The results, as a table

Render the three readers and their footprints as a `tabulate` table.

In [11]:
# Present the comparison as a report-ready table.
from tabulate import tabulate

# One row per reader; the size column comes from sys.getsizeof().
rows = [
    ["readlines() list", f"{len(lines_list):,} lines", f"{list_size:,} bytes"],
    ["LogReader class", "chunk of 1000", f"{sys.getsizeof(LogReader('data/server.log')):,} bytes"],
    ["read_log_lines generator", "1 line at a time", f"{gen_size:,} bytes"],
]
print(tabulate(rows, headers=["Approach", "Lines in memory", "Object size"], tablefmt="grid"))

+--------------------------+-------------------+-----------------+
| Approach                 | Lines in memory   | Object size     |
+==========================+===================+=================+
| readlines() list         | 140,610 lines     | 1,140,568 bytes |
+--------------------------+-------------------+-----------------+
| LogReader class          | chunk of 1000     | 48 bytes        |
+--------------------------+-------------------+-----------------+
| read_log_lines generator | 1 line at a time  | 232 bytes       |
+--------------------------+-------------------+-----------------+


## Section 10 - When a generator is the wrong tool

Generators are single-pass with no `len()` or indexing - materialize a list when you need random access or repeated passes.

In [12]:
# A generator has no len() or indexing, and it is single-pass.
gen = read_log_lines("data/server.log")
try:
    gen[0]
except TypeError as e:
    print("No indexing:", e)

n = sum(1 for _ in gen)
print("First pass consumed all", n, "lines; second pass is empty:", sum(1 for _ in gen))

lines = list(read_log_lines("data/server.log"))
print("list() re-reads it:", len(lines), "lines:", lines[0].strip())


No indexing: 'generator' object is not subscriptable
First pass consumed all 140610 lines; second pass is empty: 0
list() re-reads it: 140610 lines: 2026-08-09 00:00:00 GET /api/orders 200


## Section 11 - Optional Exercise

Rewrite the generator to yield chunks of lines instead of single lines, then re-count and confirm the totals.